
---

# ***`MLflow for Experirment Tracking`***

---

1. Create a conda environment and install mlflow in it using

    ```python
    pip install mlflow
    ```

    - Once it get's install go to terminal and type
        ```python
        mlflow ui
        ```
    - This will create a `.mlruns` folder in your system with help of which you can track activities performed by mlflow.

        - You will see a localhost link on which you can see beautiful mlflow ui.

2. Create a `src` folder and in that folder make an `local_experiment.py` file that we will use for experiements. Here is the code:

    ```python
    # import libraries
    import os
    import mlflow
    import mlflow.sklearn
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    from sklearn.datasets import load_wine
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        accuracy_score,
        confusion_matrix,
        classification_report,
    )

    # set tracking link explicitly
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

    # load the wine dataset
    wine = load_wine()
    X = wine.data
    y = wine.target

    # train test split the data
    X_train,X_test,y_train,y_test = train_test_split(
        X,
        y,
        test_size=0.10,
        random_state=42,
    )

    # define params for model
    max_depth = 10
    n_estimators = 5

    # set mlflow experiment
    mlflow.set_experiment("MLflow MLOPS Experiment")

    # mlflow tracking code
    with mlflow.start_run(run_name="1st MLflow Run"):
        
        rf = RandomForestClassifier(
            max_depth=max_depth,
            n_estimators=n_estimators,
            random_state=42,
        )
        rf.fit(X_train,y_train)
        y_pred = rf.predict(X_test)
        
        #  Accuracy
        accuracy = accuracy_score(y_test,y_pred)
        print(accuracy)
        mlflow.log_metric("Accuracy" , accuracy)

        # Generate the classification report
        clf_report = classification_report(y_test, y_pred)
        print("\nClassification Report:\n", clf_report)

        # Save the classification report as a text file
        clf_report_path = "output_files/classification_report.txt"
        with open(clf_report_path, "w") as f:
            f.write(clf_report)

        # Log the report as an artifact
        mlflow.log_artifact(clf_report_path)

        # confusion matrix
        cfm = confusion_matrix(y_test,y_pred)
        plt.figure(figsize=(8,6))
        sns.heatmap(cfm,
            annot=True,
            fmt="d",
            cmap='viridis',
            xticklabels=wine.target_names,
            yticklabels=wine.target_names,
        )
        plt.xlabel("Actual")
        plt.ylabel("Predicted")
        plt.title("Confusion Matrix")
        # save the plot
        plt.savefig("output_files/Confusion Matrix.png")

        # Log the report as an artifact
        mlflow.log_artifact("output_files/Confusion Matrix.png")

        # log code that you are currently working with
        mlflow.log_artifact(__file__)

        # log params to mlflow
        mlflow.log_param("Max Depth",max_depth)
        mlflow.log_param("n_estimators",n_estimators)

        # add tags
        mlflow.set_tags({"Author":"Naeem","Project":"Wine Classification Using Random Forest"})

        # log the model
        mlflow.sklearn.log_model(rf, "Random Forest Classifier")
        ```

- Run this file using this command:

    ```bash
    python src/mlflow_local_experiment.py
    ```

3. Make another file for dagshub tracking (remote mlflow server), let's name it `dagshub_experiment.py`.

    - First set up your github repository in dagshub.
    - Then we need `Mlflow tracking remote url` which may look like this
        ```python 
        https://dagshub.com/muhammadadilnaeem/MlFlow-Toturial.mlflow
        ``` 
    - Now we need to come back to code editor and install
        ```python
        pip install dagshub
        ```
    - then you will get this code from remote button's experiment section 
        
        ```python
        import dagshub
        dagshub.init(repo_owner='muhammadadilnaeem', repo_name='MlFlow-Toturial', mlflow=True)

        import mlflow
        with mlflow.start_run():
        mlflow.log_param('parameter name', 'value')
        mlflow.log_metric('metric name', 1)
        ``` 
    
    - Now we will add this code in `dagshub_experiment.py`:

        ```python  
        # import libraries
        import os
        import mlflow
        import dagshub
        import mlflow.sklearn
        import pandas as pd
        import seaborn as sns
        import matplotlib.pyplot as plt
        from sklearn.datasets import load_wine
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.model_selection import train_test_split
        from sklearn.metrics import (
            accuracy_score,
            confusion_matrix,
            classification_report,
        )

        # set dagshub tracking uri
        dagshub.init(repo_owner='muhammadadilnaeem', repo_name='MlFlow-Toturial', mlflow=True)
        mlflow.set_tracking_uri("https://dagshub.com/muhammadadilnaeem/MlFlow-Toturial.mlflow")

        # load the wine dataset
        wine = load_wine()
        X = wine.data
        y = wine.target

        # train test split the data
        X_train,X_test,y_train,y_test = train_test_split(
            X,
            y,
            test_size=0.10,
            random_state=42,
        )

        # define params for model
        max_depth = 15
        n_estimators = 10

        # set mlflow experiment
        mlflow.set_experiment("MLflow MLOPS Dagshub Experiment")

        # mlflow tracking code
        with mlflow.start_run(run_name="1st MLflow Dagshub Run"):
            
            rf = RandomForestClassifier(
                max_depth=max_depth,
                n_estimators=n_estimators,
                random_state=42,
            )
            rf.fit(X_train,y_train)
            y_pred = rf.predict(X_test)
            
            #  Accuracy
            accuracy = accuracy_score(y_test,y_pred)
            print(accuracy)
            mlflow.log_metric("Accuracy" , accuracy)

            # Generate the classification report
            clf_report = classification_report(y_test, y_pred)
            print("\nClassification Report:\n", clf_report)

            # Save the classification report as a text file
            clf_report_path = "output_files/classification_report.txt"
            with open(clf_report_path, "w") as f:
                f.write(clf_report)

            # Log the report as an artifact
            mlflow.log_artifact(clf_report_path)

            # confusion matrix
            cfm = confusion_matrix(y_test,y_pred)
            plt.figure(figsize=(8,6))
            sns.heatmap(cfm,
                annot=True,
                fmt="d",
                cmap='viridis',
                xticklabels=wine.target_names,
                yticklabels=wine.target_names,
            )
            plt.xlabel("Actual")
            plt.ylabel("Predicted")
            plt.title("Confusion Matrix")
            # save the plot
            plt.savefig("output_files/Confusion Matrix.png")

            # Log the report as an artifact
            mlflow.log_artifact("output_files/Confusion Matrix.png")

            # log code that you are currently working with
            mlflow.log_artifact(__file__)

            # log params to mlflow
            mlflow.log_param("Max Depth",max_depth)
            mlflow.log_param("n_estimators",n_estimators)

            # add tags
            mlflow.set_tags({"Author":"Naeem","Project":"Wine Classification Using Random Forest"})

            # log the model
            mlflow.sklearn.log_model(rf, "Random Forest Classifier")

        ``` 
- Run this file using this command:

    ```bash
    python src/mlflow_dagshub_experiment.py
    ```

- You can see the results on mlflow remote server.

4. Mlflow provides functionality to autolog necessory information instead of explicitly specifying. We will create a `mlflow_autolog.py` that will cntain this implementation.

    - here is 
        ```python
        # import libraries
        import os
        import mlflow
        import dagshub
        import mlflow.sklearn
        import pandas as pd
        import seaborn as sns
        import matplotlib.pyplot as plt
        from sklearn.datasets import load_wine
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.model_selection import train_test_split
        from sklearn.metrics import (
            accuracy_score,
            confusion_matrix,
            classification_report,
        )

        # set dagshub tracking uri
        dagshub.init(repo_owner='muhammadadilnaeem', repo_name='MlFlow-Toturial', mlflow=True)
        mlflow.set_tracking_uri("https://dagshub.com/muhammadadilnaeem/MlFlow-Toturial.mlflow")

        # load the wine dataset
        wine = load_wine()
        X = wine.data
        y = wine.target

        # train test split the data
        X_train,X_test,y_train,y_test = train_test_split(
            X,
            y,
            test_size=0.10,
            random_state=42,
        )

        # define params for model
        max_depth = 15
        n_estimators = 10

        # autolog to log information about this classification
        mlflow.autolog()

        # set mlflow experiment
        mlflow.set_experiment("MLflow MLOPS Dagshub Experiments")

        # mlflow tracking code
        with mlflow.start_run(run_name="Autolog MLflow Dagshub Run"):
            
            rf = RandomForestClassifier(
                max_depth=max_depth,
                n_estimators=n_estimators,
                random_state=42,
            )
            rf.fit(X_train,y_train)
            y_pred = rf.predict(X_test)
            
            #  Accuracy
            accuracy = accuracy_score(y_test,y_pred)
            print(accuracy)

            # Generate the classification report
            clf_report = classification_report(y_test, y_pred)
            print("\nClassification Report:\n", clf_report)

            # Save the classification report as a text file
            clf_report_path = "output_files/classification_report.txt"
            with open(clf_report_path, "w") as f:
                f.write(clf_report)

            # confusion matrix
            cfm = confusion_matrix(y_test,y_pred)
            plt.figure(figsize=(8,6))
            sns.heatmap(cfm,
                annot=True,
                fmt="d",
                cmap='viridis',
                xticklabels=wine.target_names,
                yticklabels=wine.target_names,
            )
            plt.xlabel("Actual")
            plt.ylabel("Predicted")
            plt.title("Confusion Matrix")
            # save the plot
            plt.savefig("output_files/Confusion Matrix.png")

            # log code that you are currently working with (not autologed mestion explicitly)
            mlflow.log_artifact(__file__)

            # add tags
            mlflow.set_tags({"Author":"Muhammad Adil Naeem","Project":"Wine Classification Using Random Forest"})
        ```

- Run this file using this command:

    ```bash
    python src/mlflow_autolog.py
    ```

- You can see the results on mlflow remote server.

5. Mlflow can also help us with hyperparameter tuning. Here we have an exaple code to pperform this job


    ```python
    import os
    import mlflow
    import pandas as pd
    from sklearn.datasets import load_breast_cancer
    from sklearn.model_selection import (
        train_test_split,
        GridSearchCV,
    )
    from sklearn.ensemble import RandomForestClassifier

    # load the data
    data = load_breast_cancer()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = pd.Series(data.target, name="target")

    # Splitting into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=0.2, 
        random_state=42,
    )

    # Creating the RandomForestClassifier model
    rf = RandomForestClassifier(random_state=42)

    # Defining the parameter grid for GridSearchCV
    param_grid = {
        'n_estimators': [10, 50, 100],
        'max_depth': [None, 10, 20, 30]
    }

    # Applying GridSearchCV
    grid_search = GridSearchCV(
        estimator=rf, 
        param_grid=param_grid, 
        cv=5, n_jobs=-1, 
        verbose=2
    )

    mlflow.set_experiment("Hyperparameter Tuning Experiment")

    with mlflow.start_run(run_name="Mlflow experiment using Sklearn Breast Cancer Data") as parent:
        grid_search.fit(X_train, y_train)

        # log all the child runs
        for i in range(len(grid_search.cv_results_['params'])):

            with mlflow.start_run(nested=True) as child:
                mlflow.log_params(grid_search.cv_results_["params"][i])
                mlflow.log_metric("accuracy", grid_search.cv_results_["mean_test_score"][i])

        # Displaying the best parameters and the best score
        best_params = grid_search.best_params_
        best_score = grid_search.best_score_

        # Log params
        mlflow.log_params(best_params)

        # Log metrics
        mlflow.log_metric("accuracy", best_score)

        # Log training data
        train_df = X_train.copy()
        train_df['target'] = y_train

        train_df = mlflow.data.from_pandas(train_df)
        mlflow.log_input(train_df, "training")

        # Log test data
        test_df = X_test.copy()
        test_df['target'] = y_test

        test_df = mlflow.data.from_pandas(test_df)
        mlflow.log_input(test_df, "testing")

        # Log source code
        mlflow.log_artifact(__file__)

        # Log the best model
        mlflow.sklearn.log_model(grid_search.best_estimator_, "random_forest")

        # Set tags
        mlflow.set_tag("author", "Muhammad Adil Naeem")

        print(best_params)
        print(best_score)
    ```

- Run this file using this command:

    ```bash
    python src/mlflow_hyperparameter_tuning.py
    ```

- You can see the results on mlflow remote server.

---